In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\335594\OneDrive - NTT DATA, Inc\Desktop\Hackathon\dbiz


In [5]:
from src.providers.llm.factory import (
    get_llm_provider,
)

from src.prompt_template.prompts import (
    SYSTEM_PROMPT,
    build_user_prompt,
)

In [7]:
# configure logging
from src.observability.logging_config import configure_logging

configure_logging()

2026-09-20 00:32:33,357 | INFO | src.observability.logging_config | Logging configured | level=INFO


In [12]:
from src.providers.embeddings.factory import get_embedding_provider
from src.providers.vector_store.factory import get_vector_store

In [10]:
# initialize embedding provider
embedding_provider = get_embedding_provider()

print(
    "Embedding dimension:",
    embedding_provider.dimension
)

2026-09-20 00:33:24,492 | INFO | src.providers.embeddings.factory | Creating embedding provider | provider=azure_openai
2026-09-20 00:33:24,623 | INFO | src.providers.embeddings.azure_openai | Azure embedding provider initialized | dimension=3072


Embedding dimension: 3072


In [13]:
# initialize Azure AI Search
vector_store = get_vector_store(
    embedding_dimension=embedding_provider.dimension
)

print(
    "Vector store:",
    type(vector_store).__name__
)

2026-09-20 00:35:12,155 | INFO | src.providers.vector_store.factory | Creating vector store | provider=azure_search
2026-09-20 00:35:13,101 | INFO | src.providers.vector_store.azure_search | Azure AI Search vector store initialized | index=rag-index | dimension=3072


Vector store: AzureAISearchVectorStore


In [14]:
query = (
    "What were Apple's total net sales "
    "for the three months ended June 25, 2022?"
)

query_vector = (
    embedding_provider.embed_query(
        query
    )
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=5,
)

2026-09-20 00:35:34,735 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:35:34,740 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=19.342
2026-09-20 00:35:34,772 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:35:37,231 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=2.452
2026-09-20 00:35:37,235 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5


In [15]:
user_prompt = build_user_prompt(
    query=query,
    retrieved_chunks=results,
)

In [18]:
print(user_prompt[:5000])

RETRIEVED CONTEXT

[CONTEXT 1]
Chunk ID: 2022_Q3_AAPL_p10_c1
Source: 2022 Q3 AAPL.pdf
Page: 10

Note 2 – Revenue
Net sales disaggregated by significant products and services for the three- and nine-month periods ended June 25, 2022 and June 26, 2021 were as follows (in
millions):
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
iPhone
$
40,665 
$
39,570 
$
162,863 
$
153,105 
Mac 
7,382 
8,235 
28,669 
26,012 
iPad
7,224 
7,368 
22,118 
23,610 
Wearables, Home and Accessories 
8,084 
8,775 
31,591 
29,582 
Services 
19,604 
17,486 
58,941 
50,148 
Total net sales 
$
82,959 
$
81,434 
$
304,182 
$
282,457 
(1)
Products net sales include amortization of the deferred value of unspecified software upgrade rights, which are bundled in the sales price of the respective
product.
(2)
Wearables, Home and Accessories net sales include sales of AirPods , Apple TV , Apple Watch , Beats products, HomePod mini and accessories.
(3)
Services net sales includ

In [19]:
llm_provider = get_llm_provider()

2026-09-20 00:36:39,565 | INFO | src.providers.llm.factory | Creating LLM provider | provider=azure_openai
2026-09-20 00:36:39,575 | INFO | src.providers.llm.azure_openai | Azure OpenAI LLM provider initialized | deployment=gpt-6-astra


In [21]:
response = (
    llm_provider.generate_structured_response(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
    )
)

2026-09-20 00:38:57,375 | INFO | src.providers.llm.azure_openai | Generating structured RAG response
2026-09-20 00:39:04,344 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/gpt-6-astra/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:39:04,415 | INFO | src.observability.tracing | Trace completed | step=azure_llm_structured_generation | duration_seconds=7.034
2026-09-20 00:39:04,422 | INFO | src.providers.llm.azure_openai | Structured RAG response generated | insufficient_context=False | confidence=high


In [22]:
response

RAGResponse(answer="Apple's total net sales for the three months ended June 25, 2022 were $82,959 million.", evidence=[EvidenceItem(statement='Total net sales were $82,959 million for the three months ended June 25, 2022.', chunk_id='2022_Q3_AAPL_p4_c1')], sources=[SourceItem(document='2022 Q3 AAPL.pdf', page=4, chunk_id='2022_Q3_AAPL_p4_c1')], confidence='high', insufficient_context=False)

In [23]:
print(
    response.model_dump_json(
        indent=2
    )
)

{
  "answer": "Apple's total net sales for the three months ended June 25, 2022 were $82,959 million.",
  "evidence": [
    {
      "statement": "Total net sales were $82,959 million for the three months ended June 25, 2022.",
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "sources": [
    {
      "document": "2022 Q3 AAPL.pdf",
      "page": 4,
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "confidence": "high",
  "insufficient_context": false
}


In [26]:
# test 2:
query = (
    "What was Apple's total revenue in fiscal year 2005?"
)

query_vector = (
    embedding_provider.embed_query(
        query
    )
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=5,
)

user_prompt = build_user_prompt(
    query=query,
    retrieved_chunks=results,
)

llm_provider = get_llm_provider()

response = (
    llm_provider.generate_structured_response(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
    )
)

print(
    response.model_dump_json(
        indent=2
    )
)

2026-09-20 00:43:29,241 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:43:29,282 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.893
2026-09-20 00:43:29,316 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:43:31,059 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.736
2026-09-20 00:43:31,063 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:43:31,066 | INFO | src.providers.llm.factory | Creating LLM provider | provider=azure_openai
2026-09-20 00:43:31,081 | INFO | src.providers.llm.azure_openai | Azure OpenAI LLM provider initialized | deployment=gpt-6-astra
2026-09-20 00:43:31,092 | INFO | src.providers.llm

{
  "answer": "The supplied documents do not contain Apple's total revenue for fiscal year 2005; they provide financial information for later periods and are insufficient to answer the question.",
  "evidence": [],
  "sources": [],
  "confidence": "low",
  "insufficient_context": true
}


In [27]:
from src.pipeline_factory import (
    create_rag_pipeline,
)

rag = create_rag_pipeline(
    top_k=5
)

2026-09-20 00:53:43,370 | INFO | src.pipeline_factory | Creating RAG pipeline
2026-09-20 00:53:43,373 | INFO | src.providers.embeddings.factory | Creating embedding provider | provider=azure_openai
2026-09-20 00:53:43,379 | INFO | src.providers.embeddings.azure_openai | Azure embedding provider initialized | dimension=3072
2026-09-20 00:53:43,383 | INFO | src.providers.vector_store.factory | Creating vector store | provider=azure_search
2026-09-20 00:53:43,389 | INFO | src.providers.vector_store.azure_search | Azure AI Search vector store initialized | index=rag-index | dimension=3072
2026-09-20 00:53:43,392 | INFO | src.providers.llm.factory | Creating LLM provider | provider=azure_openai
2026-09-20 00:53:43,415 | INFO | src.providers.llm.azure_openai | Azure OpenAI LLM provider initialized | deployment=gpt-6-astra
2026-09-20 00:53:43,420 | INFO | src.rag_pipeline | RAG pipeline initialized | top_k=5
2026-09-20 00:53:43,423 | INFO | src.pipeline_factory | RAG pipeline created successf

In [28]:
response = rag.ask(
    "What were Apple's total net sales "
    "for the three months ended June 25, 2022?"
)

2026-09-20 00:53:53,233 | INFO | src.rag_pipeline | Processing RAG query
2026-09-20 00:53:55,508 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:53:55,511 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=2.263
2026-09-20 00:53:55,513 | INFO | src.observability.tracing | Trace completed | step=rag_query_embedding | duration_seconds=2.264
2026-09-20 00:53:55,544 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:53:59,695 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=4.149
2026-09-20 00:53:59,700 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:53:59,702 | INFO | src.observability.tracing | Trace completed | step=

In [29]:
print(
    response.model_dump_json(
        indent=2
    )
)

{
  "answer": "Apple's total net sales for the three months ended June 25, 2022 were $82,959 million.",
  "evidence": [
    {
      "statement": "Apple reported total net sales of $82,959 million for the three months ended June 25, 2022.",
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "sources": [
    {
      "document": "2022 Q3 AAPL.pdf",
      "page": 4,
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "confidence": "high",
  "insufficient_context": false
}


In [30]:
queries = [
    "What were Apple's total net sales for the three months ended June 25, 2022?",
    "What did NVIDIA report about Data Center revenue?",
    "What risks did Intel discuss?",
    "What was Apple's revenue in fiscal year 2005?",
]

In [31]:
for query in queries:

    print("=" * 100)
    print("QUESTION:", query)

    response = rag.ask(query)

    print(
        response.model_dump_json(
            indent=2
        )
    )

2026-09-20 00:55:35,457 | INFO | src.rag_pipeline | Processing RAG query


QUESTION: What were Apple's total net sales for the three months ended June 25, 2022?


2026-09-20 00:55:36,457 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:55:36,460 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.982
2026-09-20 00:55:36,461 | INFO | src.observability.tracing | Trace completed | step=rag_query_embedding | duration_seconds=0.983
2026-09-20 00:55:36,489 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:55:39,594 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=3.102
2026-09-20 00:55:39,597 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:55:39,599 | INFO | src.observability.tracing | Trace completed | step=rag_retrieval | duration_seconds=3.137
2026-09-20 00:55:39,604 | INFO | s

{
  "answer": "Apple's total net sales for the three months ended June 25, 2022 were $82,959 million.",
  "evidence": [
    {
      "statement": "Apple reported total net sales of $82,959 million for the three months ended June 25, 2022.",
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "sources": [
    {
      "document": "2022 Q3 AAPL.pdf",
      "page": 4,
      "chunk_id": "2022_Q3_AAPL_p4_c1"
    }
  ],
  "confidence": "high",
  "insufficient_context": false
}
QUESTION: What did NVIDIA report about Data Center revenue?


2026-09-20 00:55:46,782 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:55:46,788 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.005
2026-09-20 00:55:46,791 | INFO | src.observability.tracing | Trace completed | step=rag_query_embedding | duration_seconds=1.008
2026-09-20 00:55:46,847 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:55:49,842 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=2.992
2026-09-20 00:55:49,846 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:55:49,848 | INFO | src.observability.tracing | Trace completed | step=rag_retrieval | duration_seconds=3.056
2026-09-20 00:55:49,850 | INFO | s

{
  "answer": "NVIDIA reported Data Center revenue of $14.51 billion for the third quarter of fiscal year 2024, up 279% from a year ago and up 41% from the previous quarter. Growth was driven by strong demand for the NVIDIA HGX platform for training and inferencing large language models, recommendation engines, and generative AI applications. Cloud service providers drove roughly half of Data Center revenue, with consumer internet companies and enterprises comprising approximately the other half.",
  "evidence": [
    {
      "statement": "Third-quarter fiscal year 2024 Data Center revenue was $14.51 billion, up 279% year-over-year and 41% from the previous quarter.",
      "chunk_id": "2023_Q3_NVDA_p28_c1"
    },
    {
      "statement": "Strong HGX platform sales were driven by global demand for large language model training and inferencing, recommendation engines, and generative AI applications.",
      "chunk_id": "2023_Q3_NVDA_p28_c1"
    },
    {
      "statement": "Cloud service

2026-09-20 00:56:00,790 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:56:00,794 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.006
2026-09-20 00:56:00,796 | INFO | src.observability.tracing | Trace completed | step=rag_query_embedding | duration_seconds=1.008
2026-09-20 00:56:00,836 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:56:04,586 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=3.748
2026-09-20 00:56:04,589 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:56:04,591 | INFO | src.observability.tracing | Trace completed | step=rag_retrieval | duration_seconds=3.793
2026-09-20 00:56:04,593 | INFO | s

{
  "answer": "Intel discussed risks involving:\n- Cybersecurity, privacy, and potential security vulnerabilities in its products.\n- Investments and transactions.\n- Intellectual property, litigation, and regulatory proceedings.\n- Evolving regulatory and legal requirements across jurisdictions.\n- Geopolitical and international trade conditions, including Russia's war on Ukraine, recent events in Israel, and rising tensions between the US and China.\n- Debt obligations and access to capital.\n- Large-scale global operations.\n- Macroeconomic conditions, including regional or global downturns or recessions.\n- COVID-19 or similar pandemics.\n\nIntel also described specific legal matters involving European Commission competition proceedings and lawsuits related to Spectre, Meltdown, and other security vulnerabilities.",
  "evidence": [
    {
      "statement": "Intel listed cybersecurity and privacy, investment and transaction, IP and litigation, evolving legal requirements, geopolitic

2026-09-20 00:56:18,080 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:56:18,091 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.481
2026-09-20 00:56:18,096 | INFO | src.observability.tracing | Trace completed | step=rag_query_embedding | duration_seconds=1.487
2026-09-20 00:56:18,175 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:56:22,270 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=4.090
2026-09-20 00:56:22,272 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5
2026-09-20 00:56:22,274 | INFO | src.observability.tracing | Trace completed | step=rag_retrieval | duration_seconds=4.176
2026-09-20 00:56:22,277 | INFO | s

{
  "answer": "The supplied documents are insufficient to determine Apple's revenue in fiscal year 2005; they provide financial results for periods in 2021 and 2022, not fiscal year 2005.",
  "evidence": [
    {
      "statement": "The revenue table covers the three months ended December 31, 2022 and December 25, 2021.",
      "chunk_id": "2023_Q1_AAPL_p10_c1"
    },
    {
      "statement": "The revenue table covers the three- and nine-month periods ended June 25, 2022 and June 26, 2021.",
      "chunk_id": "2022_Q3_AAPL_p10_c1"
    }
  ],
  "sources": [
    {
      "document": "2023 Q1 AAPL.pdf",
      "page": 10,
      "chunk_id": "2023_Q1_AAPL_p10_c1"
    },
    {
      "document": "2022 Q3 AAPL.pdf",
      "page": 10,
      "chunk_id": "2022_Q3_AAPL_p10_c1"
    }
  ],
  "confidence": "low",
  "insufficient_context": true
}
